### Parameters

In [0]:
dbutils.widgets.text('Incremental_Flag','0')

In [0]:
# 0 = Full Load
# 1 = Incremental Load

Incremental_Flag = dbutils.widgets.get('Incremental_Flag')
print("Incremental_Flag = ", Incremental_Flag)

### Create Dimensions

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://silver@adlscarproject.dfs.core.windows.net/carsales")

df.display()

### Catalog creation
Create a db in bronze instead of querying directly from the Data lake
It's better for users to navigate on bronze

In [0]:
%sql
-- Create catalog Car_Project



### Database creation

In [0]:
%sql
Create database if not exists Car_Project.Silver
Comment 'Silver Layer Medallion Archtecture'
MANAGED LOCATION 'abfss://silver@adlscarproject.dfs.core.windows.net/database/'



In [0]:
%sql
Create database if not exists Car_Project.Gold
Comment 'Gold Layer Medallion Archtecture'
MANAGED LOCATION 'abfss://gold@adlscarproject.dfs.core.windows.net/database/'



### Table in Silver later creation

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# COPYING BRONZE SCHEMA
df_bronze = spark.read.format("parquet")\
    .option("inferSchema", "true")\
    .option("header", "true")\
    .load("abfss://bronze@adlscarproject.dfs.core.windows.net/rawdata")

# ADDING AUTO INCREMENT
df_silver = df_bronze.withColumn("Model_Key", monotonically_increasing_id())
df_silver = df_bronze.withColumn("Model_Category",split(df_silver['Model_ID'], '_').getItem(0))




# CREATING SILVER TABLE
df_silver.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("Car_Project.Silver.Models")

In [0]:
%sql
Select * from car_project.silver.models limit 6
    

In [0]:
%sql 
-- SHOW TABLES IN Car_Project.Gold
-- DESCRIBE Car_Project.Gold.Dim_Model;


In [0]:
# DIM CREATION - 

if spark.catalog.tableExists("Car_Project.gold.Dim_Models"):

    df_sink = spark.sql("""
                    SELECT Dim_Model_Key, Model_ID, Model_Category 
                    FROM Car_Project.Silver.Models
                """)
else: 
        df_sink = spark.sql("""
                    SELECT 1 as Dim_Model_Key, Model_ID, Model_Category 
                    FROM Car_Project.Silver.Models
                    Where 1=0
                """)
    


In [0]:
#JOIN 

df_filter = df_src.join(df_sink, df_src.Model_ID == df_sink.Model_ID, "left").select(df_src.Model_ID, df_src.Model_Category, df_sink.Dim_Model_Key)
df_filter.display()

In [0]:
df_filter_old = df_filter.filter(df_filter.Dim_Model_Key.isNotNull())
df_filter_old.display()

In [0]:
df_filter_new = df_filter.filter(df_filter.Dim_Model_Key.isNull()).select("Model_ID", "Model_Category")
df_filter_new.display()